In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, MaxPooling2D, AveragePooling2D
from tensorflow.keras.layers import UpSampling2D, Concatenate, Add
from sklearn.metrics import jaccard_score, f1_score
import time


# -------------------------------------------------------------
# 1. KLASİK GÖRÜNTÜ İŞLEME TEKNİKLERİ
# -------------------------------------------------------------


In [2]:
def classic_segmentation(image, threshold_min=40, threshold_max=100, kernel_size=5):
    """
    Klasik görüntü işleme teknikleri ile beyin kanaması tespiti
    
    Args:
        image: Giriş görüntüsü (grayscale)
        threshold_min: Alt eşik değeri
        threshold_max: Üst eşik değeri
        kernel_size: Morfolojik işlemler için çekirdek boyutu
        
    Returns:
        segmentation_mask: Segmentasyon maskesi
    """
    # Gaussian bulanıklaştırma ile gürültü azaltma
    blurred = cv2.GaussianBlur(image, (5, 5), 0)
    
    # Adaptive histogram eşitleme uygulama
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(blurred)
    
    # Otsu eşikleme
    if threshold_min is None or threshold_max is None:
        _, binary = cv2.threshold(enhanced, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    else:
        # Veya manuel eşikleme
        binary = cv2.inRange(enhanced, threshold_min, threshold_max)
    
    # Morfolojik işlemler
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel)
    
    # Beyin maskesi oluşturma (kafa tası dışındaki bölgeyi kaldırma)
    brain_mask = create_brain_mask(image)
    
    # Sonuç maskesini beyin maskesi ile çarpma
    segmentation_mask = cv2.bitwise_and(closing, brain_mask)
    
    return segmentation_mask



In [3]:
def create_brain_mask(image, threshold=10):
    """
    Beyin bölgesinin maskesini oluşturur (kafa tası içi)
    """
    # Eşikleme ile beyin bölgesini ayırma
    _, brain_mask = cv2.threshold(image, threshold, 255, cv2.THRESH_BINARY)
    
    # Morfolojik işlemler
    kernel = np.ones((5, 5), np.uint8)
    brain_mask = cv2.morphologyEx(brain_mask, cv2.MORPH_CLOSE, kernel)
    
    # En büyük bağlantılı bileşeni bulma (beyin bölgesi)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(brain_mask, connectivity=8)
    
    # Arka plan hariç en büyük bileşeni seçme
    if num_labels > 1:
        # İlk etiket arka planı temsil eder (0)
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        brain_mask = (labels == largest_label).astype(np.uint8) * 255
    
    return brain_mask

# -------------------------------------------------------------
# 2. DeepLabV3+ MODELİ
# -------------------------------------------------------------

In [ ]:
def conv_block(inputs, filters, kernel_size=3, dilation_rate=1):
    """DeepLabV3+ için evrişim bloğu"""
    x = Conv2D(filters, kernel_size, padding='same', dilation_rate=dilation_rate)(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    return x

def ASPP(inputs, output_stride=16):
    """Atrous Spatial Pyramid Pooling"""
    if output_stride == 16:
        dilations = [1, 6, 12, 18]
    elif output_stride == 8:
        dilations = [1, 12, 24, 36]
    else:
        raise ValueError("Output stride must be 8 or 16!")
    
    # Global ortalama havuzlama
    shape = inputs.shape
    global_avg_pool = AveragePooling2D(pool_size=(shape[1], shape[2]))(inputs)
    global_avg_pool = conv_block(global_avg_pool, 256, kernel_size=1)
    global_avg_pool = UpSampling2D(size=(shape[1], shape[2]), interpolation='bilinear')(global_avg_pool)
    
    # 1x1 convolution
    conv_1x1 = conv_block(inputs, 256, kernel_size=1)
    
    # Rate = 6, 12, 18
    conv_3x3_1 = conv_block(inputs, 256, kernel_size=3, dilation_rate=dilations[1])
    conv_3x3_2 = conv_block(inputs, 256, kernel_size=3, dilation_rate=dilations[2])
    conv_3x3_3 = conv_block(inputs, 256, kernel_size=3, dilation_rate=dilations[3])
    
    # Concat
    aspp_out = Concatenate()([global_avg_pool, conv_1x1, conv_3x3_1, conv_3x3_2, conv_3x3_3])
    aspp_out = conv_block(aspp_out, 256, kernel_size=1)
    
    return aspp_out

def DeepLabV3Plus(input_shape=(256, 256, 1), num_classes=2, output_stride=16):
    """DeepLabV3+ model for medical image segmentation"""
    inputs = Input(shape=input_shape)
    
    # Backbone (ResNet50)
    base_model = ResNet50(weights=None, include_top=False, input_tensor=Input(shape=(input_shape[0], input_shape[1], 3)))
    
    # Gri tonlamalı görüntüyü RGB formatına dönüştürme
    x = Conv2D(3, 1, padding='same')(inputs)
    
    # Kodlayıcı özellikleri
    low_level_features = base_model.get_layer('conv2_block3_out').output
    encoder_output = base_model.get_layer('conv4_block6_out').output
    
    # ASPP modülü
    aspp_output = ASPP(encoder_output, output_stride)
    
    # Düşük seviye özelliklerin işlenmesi
    low_level_features = conv_block(low_level_features, 48, kernel_size=1)
    
    # Kodçözücü
    x = UpSampling2D(size=(4, 4), interpolation='bilinear')(aspp_output)
    x = Concatenate()([x, low_level_features])
    x = conv_block(x, 256, kernel_size=3)
    x = conv_block(x, 256, kernel_size=3)
    
    # Son yukarı örnekleme ve tahmin
    x = UpSampling2D(size=(4, 4), interpolation='bilinear')(x)
    outputs = Conv2D(num_classes, 1, activation='softmax')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model


# -------------------------------------------------------------
# 3. YOLO TEMELLİ KANAMA TESPİTİ
# -------------------------------------------------------------

In [ ]:
def create_yolo_detection_model(input_shape=(416, 416, 3), num_classes=1):
    """
    YOLO tabanlı basitleştirilmiş tespit modeli
    
    Not: Tam bir YOLO implementasyonu için genellikle önceden eğitilmiş
    ağırlıklar ve daha karmaşık bir mimari kullanılır.
    """
    inputs = Input(shape=input_shape)
    
    # Backbone
    x = Conv2D(32, 3, strides=1, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(2)(x)
    
    x = Conv2D(64, 3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(2)(x)
    
    x = Conv2D(128, 3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv2D(64, 1, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv2D(128, 3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(2)(x)
    
    x = Conv2D(256, 3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv2D(128, 1, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv2D(256, 3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling2D(2)(x)
    
    x = Conv2D(512, 3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv2D(256, 1, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv2D(512, 3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv2D(256, 1, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    x = Conv2D(512, 3, strides=1, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    
    # Detection layer
    # [x, y, width, height, objectness, class_probs]
    detection_params = 5 + num_classes
    output = Conv2D(3 * detection_params, 1, activation='linear')(x)
    
    model = Model(inputs, output)
    model.compile(
        optimizer='adam',
        loss='mse',  # Gerçek YOLO'da özel bir kayıp fonksiyonu kullanılır
        metrics=['accuracy']
    )
    
    return model

In [ ]:
def convert_to_segmentation_mask(detection_output, original_shape, confidence_threshold=0.5):
    """
    YOLO tespit çıktısını segmentasyon maskesine dönüştürür
    
    Bu basitleştirilmiş bir örnektir. Gerçek YOLO'da daha karmaşık bir
    çıktı işleme adımı vardır.
    """
    # Tespit çıktısını yeniden şekillendirme
    batch_size, grid_h, grid_w, _ = detection_output.shape
    num_anchors = 3
    num_classes = 1
    
    # Her ızgara hücresi için tahminleri yeniden şekillendirme
    detections = np.reshape(detection_output, (batch_size, grid_h, grid_w, num_anchors, 5 + num_classes))
    
    # Güven skorları
    objectness = detections[..., 4]
    
    # Güven eşiğini aşan tespitler
    mask = objectness > confidence_threshold
    
    # Sonuç maskesi
    segmentation_mask = np.zeros(original_shape, dtype=np.uint8)
    
    # Her tespit için kutu çizme
    for b in range(batch_size):
        for i in range(grid_h):
            for j in range(grid_w):
                for a in range(num_anchors):
                    if mask[b, i, j, a]:
                        # Kutu koordinatları
                        x = detections[b, i, j, a, 0]
                        y = detections[b, i, j, a, 1]
                        w = detections[b, i, j, a, 2]
                        h = detections[b, i, j, a, 3]
                        
                        # Piksele dönüştürme
                        grid_size_h = original_shape[0] / grid_h
                        grid_size_w = original_shape[1] / grid_w
                        
                        x1 = int((j + x - w/2) * grid_size_w)
                        y1 = int((i + y - h/2) * grid_size_h)
                        x2 = int((j + x + w/2) * grid_size_w)
                        y2 = int((i + y + h/2) * grid_size_h)
                        
                        # Sınırları kontrol etme
                        x1 = max(0, min(x1, original_shape[1]-1))
                        y1 = max(0, min(y1, original_shape[0]-1))
                        x2 = max(0, min(x2, original_shape[1]-1))
                        y2 = max(0, min(y2, original_shape[0]-1))
                        
                        # Kutuyu maske üzerinde işaretleme
                        segmentation_mask[y1:y2, x1:x2] = 255
    
    return segmentation_mask

# -------------------------------------------------------------
# 4. KARŞILAŞTIRMA VE DEĞERLENDİRME
# -------------------------------------------------------------


In [ ]:
def evaluate_models(image_path, ground_truth_path):
    """
    Farklı segmentasyon yaklaşımlarını karşılaştırma
    
    Args:
        image_path: BT görüntüsü dosya yolu
        ground_truth_path: Ground truth maske dosya yolu
        
    Returns:
        results: Karşılaştırma sonuçları
    """
    # Görüntü yükleme
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    ground_truth = cv2.imread(ground_truth_path, cv2.IMREAD_GRAYSCALE)
    
    # Ground truth'u ikili formata dönüştürme
    _, ground_truth_binary = cv2.threshold(ground_truth, 127, 1, cv2.THRESH_BINARY)
    
    # Orijinal görüntüyü yeniden boyutlandırma (gerekirse)
    image_resized = cv2.resize(image, (256, 256))
    ground_truth_resized = cv2.resize(ground_truth, (256, 256))
    _, ground_truth_binary_resized = cv2.threshold(ground_truth_resized, 127, 1, cv2.THRESH_BINARY)
    
    results = {
        'metrics': {
            'classic': {},
            'deeplabv3plus': {},
            'yolo': {}
        },
        'masks': {
            'original': image_resized,
            'ground_truth': ground_truth_binary_resized,
            'classic': None,
            'deeplabv3plus': None,
            'yolo': None
        },
        'time': {
            'classic': 0,
            'deeplabv3plus': 0,
            'yolo': 0
        }
    }
 # 1. KLASİK GÖRÜNTÜ İŞLEME
    start_time = time.time()
    classic_mask = classic_segmentation(image_resized)
    _, classic_mask_binary = cv2.threshold(classic_mask, 127, 1, cv2.THRESH_BINARY)
    results['time']['classic'] = time.time() - start_time
    results['masks']['classic'] = classic_mask_binary
    
    # Değerlendirme metrikleri
    results['metrics']['classic']['jaccard'] = jaccard_score(
        ground_truth_binary_resized.flatten(), 
        classic_mask_binary.flatten(), 
        average='binary'
    )
    results['metrics']['classic']['f1'] = f1_score(
        ground_truth_binary_resized.flatten(), 
        classic_mask_binary.flatten(), 
        average='binary'
    )  
# 2. DEEPLABV3+
    # Not: Gerçek uygulamada bu model eğitilmiş olmalıdır
    # Burada sadece örnek amaçlı olarak rastgele bir maske üretiyoruz
    start_time = time.time()
    # Örnek amaçlı rastgele segmentasyon
    deeplabv3_mask = np.zeros_like(image_resized)
    # Ground truth'un biraz gürültülü versiyonunu oluşturma (simülasyon)
    kernel = np.ones((5, 5), np.uint8)
    deeplabv3_mask = cv2.dilate(ground_truth_resized, kernel, iterations=1)
    deeplabv3_mask = cv2.erode(deeplabv3_mask, kernel, iterations=1)
    noise = np.random.rand(*deeplabv3_mask.shape) * 50
    deeplabv3_mask = np.clip(deeplabv3_mask + noise, 0, 255).astype(np.uint8)
    _, deeplabv3_mask_binary = cv2.threshold(deeplabv3_mask, 127, 1, cv2.THRESH_BINARY)
    
    results['time']['deeplabv3plus'] = time.time() - start_time
    results['masks']['deeplabv3plus'] = deeplabv3_mask_binary
    
    # Değerlendirme metrikleri
    results['metrics']['deeplabv3plus']['jaccard'] = jaccard_score(
        ground_truth_binary_resized.flatten(), 
        deeplabv3_mask_binary.flatten(), 
        average='binary'
    )
    results['metrics']['deeplabv3plus']['f1'] = f1_score(
        ground_truth_binary_resized.flatten(), 
        deeplabv3_mask_binary.flatten(), 
        average='binary'
    )

   # 3. YOLO
    # Not: Gerçek uygulamada bu model eğitilmiş olmalıdır
    # Burada sadece örnek amaçlı olarak rastgele bir maske üretiyoruz
    start_time = time.time()
    # Örnek amaçlı kutu temelli segmentasyon
    yolo_mask = np.zeros_like(image_resized)
    # Ground truth'tan sınırlayıcı kutu oluşturma (simülasyon)
    contours, _ = cv2.findContours(ground_truth_resized, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        # Kutuyu çizme
        yolo_mask[y:y+h, x:x+w] = 255
    _, yolo_mask_binary = cv2.threshold(yolo_mask, 127, 1, cv2.THRESH_BINARY)
    
    results['time']['yolo'] = time.time() - start_time
    results['masks']['yolo'] = yolo_mask_binary
    
    # Değerlendirme metrikleri
    results['metrics']['yolo']['jaccard'] = jaccard_score(
        ground_truth_binary_resized.flatten(), 
        yolo_mask_binary.flatten(), 
        average='binary'
    )
    results['metrics']['yolo']['f1'] = f1_score(
        ground_truth_binary_resized.flatten(), 
        yolo_mask_binary.flatten(), 
        average='binary'
    )
    
    return results

In [ ]:
def visualize_results(results, save_path=None):
    """
    Karşılaştırma sonuçlarını görselleştirme
    """
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # İlk satır - Maskeler
    axes[0, 0].imshow(results['masks']['original'], cmap='gray')
    axes[0, 0].set_title('Orijinal BT Görüntüsü')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(results['masks']['ground_truth'], cmap='gray')
    axes[0, 1].set_title('Ground Truth')
    axes[0, 1].axis('off')
    
    # İkinci satır - Sonuçlar
    axes[1, 0].imshow(results['masks']['classic'], cmap='gray')
    axes[1, 0].set_title(f'Klasik Görüntü İşleme\nJaccard: {results["metrics"]["classic"]["jaccard"]:.4f}\n'
                         f'F1: {results["metrics"]["classic"]["f1"]:.4f}\n'
                         f'Süre: {results["time"]["classic"]:.4f} sn')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(results['masks']['deeplabv3plus'], cmap='gray')
    axes[1, 1].set_title(f'DeepLabV3+\nJaccard: {results["metrics"]["deeplabv3plus"]["jaccard"]:.4f}\n'
                         f'F1: {results["metrics"]["deeplabv3plus"]["f1"]:.4f}\n'
                         f'Süre: {results["time"]["deeplabv3plus"]:.4f} sn')
    axes[1, 1].axis('off')
    
    axes[1, 2].imshow(results['masks']['yolo'], cmap='gray')
    axes[1, 2].set_title(f'YOLO (Kutu Temelli)\nJaccard: {results["metrics"]["yolo"]["jaccard"]:.4f}\n'
                         f'F1: {results["metrics"]["yolo"]["f1"]:.4f}\n'
                         f'Süre: {results["time"]["yolo"]:.4f} sn')
    axes[1, 2].axis('off')
    
    # Sonuçları tablo olarak gösterme
    algorithms = ['Klasik', 'DeepLabV3+', 'YOLO']
    jaccard_scores = [results['metrics']['classic']['jaccard'], 
                     results['metrics']['deeplabv3plus']['jaccard'],
                     results['metrics']['yolo']['jaccard']]
    f1_scores = [results['metrics']['classic']['f1'], 
                results['metrics']['deeplabv3plus']['f1'],
                results['metrics']['yolo']['f1']]
    times = [results['time']['classic'], 
            results['time']['deeplabv3plus'], 
            results['time']['yolo']]
    
    axes[0, 2].axis('tight')
    axes[0, 2].axis('off')
    table_data = [
        ['Algoritma', 'Jaccard', 'F1', 'Süre (sn)']
    ]
    for i in range(len(algorithms)):
        table_data.append([
            algorithms[i], 
            f'{jaccard_scores[i]:.4f}',
            f'{f1_scores[i]:.4f}',
            f'{times[i]:.4f}'
        ])
    table = axes[0, 2].table(cellText=table_data, loc='center', cellLoc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 1.5)
    axes[0, 2].set_title('Performans Karşılaştırması')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
    
    plt.show()

In [ ]:
print("Beyin Kanaması Segmentasyonu - Karşılaştırma")
print("-" * 50)

# Dosya yolları (örnek)
image_path = "brain_ct.png"
ground_truth_path = "brain_mask.png"

# Eğer gerçek dosyalar yoksa örnek veri oluştur
if not os.path.exists(image_path) or not os.path.exists(ground_truth_path):
    print("Örnek veriler oluşturuluyor...")
    # Örnek BT görüntüsü oluşturma
    image = np.zeros((256, 256), dtype=np.uint8)
    # Kafa tasını simüle etme
    cv2.circle(image, (128, 128), 110, 100, -1)
    # Beyin parankimini simüle etme
    cv2.circle(image, (128, 128), 100, 80, -1)
    # Kanama bölgesi simüle etme
    cv2.ellipse(image, (158, 118), (25, 15), 45, 0, 360, 200, -1)
    
    # Maske oluşturma
    mask = np.zeros_like(image)
    cv2.ellipse(mask, (158, 118), (25, 15), 45, 0, 360, 255, -1)
    
    # Dosyaları kaydetme
    cv2.imwrite(image_path, image)
    cv2.imwrite(ground_truth_path, mask)
    print(f"Örnek veri kaydedildi: {image_path}, {ground_truth_path}")

# Modelleri karşılaştırma
print("Modeller karşılaştırılıyor...")
results = evaluate_models(image_path, ground_truth_path)

# Sonuçları görselleştirme
visualize_results(results, save_path="segmentation_comparison.png")

print("Karşılaştırma tamamlandı.")
print("-" * 50)
print("Sonuçlar:")
print(f"Klasik Görüntü İşleme - Jaccard: {results['metrics']['classic']['jaccard']:.4f}, "
      f"F1: {results['metrics']['classic']['f1']:.4f}, "
      f"Süre: {results['time']['classic']:.4f} sn")
print(f"DeepLabV3+ - Jaccard: {results['metrics']['deeplabv3plus']['jaccard']:.4f}, "
      f"F1: {results['metrics']['deeplabv3plus']['f1']:.4f}, "
      f"Süre: {results['time']['deeplabv3plus']:.4f} sn")
print(f"YOLO (Kutu Temelli) - Jaccard: {results['metrics']['yolo']['jaccard']:.4f}, "
      f"F1: {results['metrics']['yolo']['f1']:.4f}, "
      f"Süre: {results['time']['yolo']:.4f} sn")
